## Create dataframes for visualization

In [54]:
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from matplotlib import pyplot as plt
from parse import parse
import numpy as np
import os 
import pandas as pd 
from hedypet.utils import get_norm_consts, get_patient_metadata
from hedypet.utils import load_splits 


# RAW = Path("/depict/data/hedit/raw/")
# DERIVATIVES = Path("/depict/data/hedit/derivatives/pipeline-bodydyn/")

def filter_and_combine_regions(df,regions):
    group_cols = ["sub","erosion","task"]
    if "tix" in df:
        group_cols+=["tix"]
    results = []

    for k, v in regions.items():
        mask = df.region.isin(v) if isinstance(v, list) else df.region == v
        filtered = df[mask]
        
        if isinstance(v, list):
            result = filtered.groupby(group_cols).apply(
                lambda x: pd.Series({
                    'mu': (x.mu * x.n).sum() / x.n.sum(),
                    'n': x.n.sum()
                })
            ).reset_index()
        else:
            result = filtered
        
        result['region'] = k
        results.append(result)
    df = pd.concat(results, ignore_index=True)
    return df.drop("ix",axis=1)

regions = {
    "Spleen":"spleen",
    "Kidneys": ["kidney_right","kidney_left"],
    "Aorta": "aorta",
    "Liver":"liver",
    "Stomach":"stomach",
    "Lungs": ['lung_upper_lobe_left', 'lung_lower_lobe_left','lung_upper_lobe_right', 'lung_middle_lobe_right','lung_lower_lobe_right'],
    'Colon':"colon",
    'Subcutaneous fat':"subcutaneous_fat",
    "Skeletal muscle":"skeletal_muscle",
    'Visceral fat':"visceral_fat",
    "Gray matter":['Left-Cerebral-Cortex', 'Right-Cerebral-Cortex'], 
    "White matter": ['Right-Cerebral-White-Matter' , 'Left-Cerebral-White-Matter'],
    "Brain":"brain",
    "Esophagus": "esophagus",
    "Trachea": "trachea",
    "Small intestine": ["small_bowel","duodenum"],
    "Skull": "skull",
    "Ribs": [f"rib_left_{i}" for i in range(1,13)] + [f"rib_right_{i}" for i in range(1,13)],
    "Urinary bladder": "urinary_bladder",
    "Gallbladder": "gallbladder",
    "Pancreas": "pancreas",
    "Heart": "heart",
    "Prostate": "prostate",
    "Thyroid": "thyroid_gland",
    "Spinal Cord": "spinal_cord",
    "Hip": ["hip_right","hip_left"],
    "Femur": ["femur_left","femur_right"],
    "Humerus": ["humerus_left","humerus_right"],
    "Vena cava": ["superior_vena_cava","inferior_vena_cava"],
    "Clavicula": ["clavicula_left","clavicula_right"],
    "Scapula": ["scapula_left","scapula_right"],
    "Vertebral column": ["sacrum","vertebrae_S1"] + [f"vertebrae_L{i}" for i in range(1,6)]+ [f"vertebrae_T{i}" for i in range(1,13)]+ [f"vertebrae_C{i}" for i in range(1,8)]
}






In [58]:
len(df)/len(df1)

7.62818153940509

In [56]:
df = pd.read_pickle("/homes/hinge/Projects/hedyPET/src/hedypet/analysis/acstatPSF_means.pkl")
df1 = filter_and_combine_regions(df,regions)
df1.to_pickle("means.pkl")

/tmp/ipykernel_2548816/1694217049.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result['region'] = k
/tmp/ipykernel_2548816/1694217049.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = filtered.groupby(group_cols).apply(
/tmp/ipykernel_2548816/1694217049.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [ ]:

df = pd.read_pickle("/homes/hinge/Projects/hedyPET/src/hedypet/analysis/acdynPSF_means.pkl")
df = filter_and_combine_regions(df,regions)
t = [1,3,5,7,9,11.0,13.0,15.0,17.0,19.0,21.0,23.0,25.0,27.0,29.0,31.0,33.0,35.0,37.0,39.0,42.5,47.5,52.5,57.5,62.5,67.5,72.5,77.5,82.5,87.5,95.0,105,115,125,135,145,155,165,175,185,195,205,215,225,235,270,330,390,450,510,570,660,780,900,1020,1140,1260,1380,1500,1620,1740,1950,2250,2550,2850,3150,3450,3750,4050]
df["t"] = df.tix.map(lambda x: t[int(x)])
df.to_pickle("tacs.pkl")

In [ ]:
# df = filter_and_combine_regions(df,regions)
# t = [1,3,5,7,9,11.0,13.0,15.0,17.0,19.0,21.0,23.0,25.0,27.0,29.0,31.0,33.0,35.0,37.0,39.0,42.5,47.5,52.5,57.5,62.5,67.5,72.5,77.5,82.5,87.5,95.0,105,115,125,135,145,155,165,175,185,195,205,215,225,235,270,330,390,450,510,570,660,780,900,1020,1140,1260,1380,1500,1620,1740,1950,2250,2550,2850,3150,3450,3750,4050]
# df["t"] = df.tix.map(lambda x: t[int(x)])
# df.to_pickle("tacs.pkl")

In [72]:
def filter_and_combine_regions_ki(df,regions):
    group_cols = ["sub","erosion","task","if_tag","frames"]
    results = []

    for k, v in regions.items():
        mask = df.region.isin(v) if isinstance(v, list) else df.region == v
        filtered = df[mask]
        
        if isinstance(v, list):
            result = filtered.groupby(group_cols).apply(
                lambda x: pd.Series({
                    'slope': (x.slope * x.n).sum() / x.n.sum(),
                    'n': x.n.sum()
                })
            ).reset_index()
        else:
            result = filtered
        
        result['region'] = k
        results.append(result)
    df = pd.concat(results, ignore_index=True)
    return df

In [ ]:
df1 = pd.read_pickle("/homes/hinge/Projects/hedyPET/src/hedypet/analysis/patlak_ki.pkl")
df2 = df1.drop(["tac_if_path","tac_organ_path","ix"],axis=1)
df2 = filter_and_combine_regions_ki(df2,regions)
df2.iloc[-50:]
df2["aorta_voi"] = df2["if_tag"].apply(lambda x: int(str(x).split("_")[-1])).map({1:"Ascending",2:"Top",3:"Descending top", 4:"Descending bottom"})
df2["voi_volume"] = df2.if_tag.apply(lambda x: "_".join(x.split("_")[1:3])).map({'ml-1_width-3':"1mL, 3px width", 'ml-1.5_width-5':"1.5mL, 5px width", 'ml-2_width-5':"2mL, 5px width"})
df2 = df2.drop(["if_tag"],axis=1)
df2.to_pickle("patlak.pkl")

/tmp/ipykernel_2548816/4285703716.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result['region'] = k
/tmp/ipykernel_2548816/4285703716.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = filtered.groupby(group_cols).apply(
/tmp/ipykernel_2548816/4285703716.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

,slope,n,frames,if_tag,sub,task,erosion,region
547798,0.003988,195933.0,8,aortavois_ml-1_width-3_0_1,sub-099,ts_total,1,Vertebral column
547799,-0.000665,195933.0,2,aortavois_ml-1_width-3_0_2,sub-099,ts_total,1,Vertebral column
547800,0.001720,195933.0,3,aortavois_ml-1_width-3_0_2,sub-099,ts_total,1,Vertebral column
547801,0.003632,195933.0,4,aortavois_ml-1_width-3_0_2,sub-099,ts_total,1,Vertebral column
547802,0.003836,195933.0,5,aortavois_ml-1_width-3_0_2,sub-099,ts_total,1,Vertebral column
547803,0.003921,195933.0,6,aortavois_ml-1_width-3_0_2,sub-099,ts_total,1,Vertebral column
547804,0.003799,195933.0,7,aortavois_ml-1_width-3_0_2,sub-099,ts_total,1,Vertebral column
547805,0.003877,195933.0,8,aortavois_ml-1_width-3_0_2,sub-099,ts_total,1,Vertebral column
547806,-0.002505,195933.0,2,aortavois_ml-1_width-3_0_3,sub-099,ts_total,1,Vertebral column
547807,0.004451,195933.0,3,aortavois_ml-1_width-3_0_3,sub-099,ts_total,1,Vertebral column


In [26]:
import pandas as pd

data = []
for sub in load_splits()["all"]:
    d = get_norm_consts(sub)
    d.update(get_patient_metadata(sub)) 
    d["sub"] = sub
    d["demographic-group"] = d["demographic-group"][1:]
    data.append(d)

df_norm = pd.DataFrame(data)
df_norm.to_pickle("norm2.pkl")

In [34]:
df = pd.read_pickle("/homes/hinge/Projects/hedyPET/src/hedypet/analysis/acdynPSF_means.pkl")
df

,mu,std,n,sub,task,erosion,ix,region
0,0.000000,0.000000,61988,sub-000,ts_total,0,1,spleen
1,0.000000,0.000000,61988,sub-000,ts_total,0,1,spleen
2,0.000000,0.000000,61988,sub-000,ts_total,0,1,spleen
3,0.000000,0.000000,61988,sub-000,ts_total,0,1,spleen
4,0.000000,0.000000,61988,sub-000,ts_total,0,1,spleen
...,...,...,...,...,...,...,...,...
64,52.485367,52.485367,124872000,sub-099,totalimage,0,1,body
65,52.542271,52.542271,124872000,sub-099,totalimage,0,1,body
66,52.505356,52.505356,124872000,sub-099,totalimage,0,1,body
67,52.581230,52.581230,124872000,sub-099,totalimage,0,1,body
